# Label YouTube Text as either Prediction or Non-Prediction

- **Tasks:**
    1. Load YouTube transcripts extracted from `1-extract_transcripts.ipynb`.
    4. Clean the data.
    5. Create prompt to identify predictions.
    6. Pass each row of text + prompt into the [UF NaviGator Toolkit](https://it.ufl.edu/ai/navigator-toolkit/) or [Groq](https://console.groq.com/home) LLMs.
    7. Perform majority vote (MV) across all LLMs.
    8. Filter by MV label = 1, hence prediction.
    9. Verify that the text is/is not a prediction and update accordingly.

In [1]:
import os
import sys

import re
from typing import List

import pandas as pd
import numpy as np

from tqdm import tqdm
from youtube_transcript_api import YouTubeTranscriptApi

# Get the current working directory of the notebook
notebook_dir = os.getcwd()
# Add the parent directory to the system path
sys.path.append(os.path.join(notebook_dir, '../'))
# from metrics import Metrics
# from feature_extraction import SpacyFeatureExtraction
from data_processing import DataProcessing
from text_generation_models import TextGenerationModelFactory, llm_classify_text

In [2]:
pd.set_option('max_colwidth', 800)
# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_rows', None)

## Load Data

In [3]:
base_data_path = DataProcessing.load_base_data_path(notebook_dir)
yt_data_path = os.path.join(base_data_path, 'yt', 'whisper_transcripts', 'sports')
transcripts = os.listdir(yt_data_path)

Choose the transcript video that you want to pass through the pipeline. 

Adjust the element from "transcripts" list to choose the right "Video ID".

In [4]:
transcript_path = os.path.join(yt_data_path, transcripts[2])

raw_transcripts_df = DataProcessing.load_from_file(path=transcript_path, file_type='csv')

raw_transcripts_df.head()

,Text,Start Time,Duration,Video ID
0,We have reached the end.,0.00,4.50,LXPQrZV4Cfw
1,One last game.,5.68,1.24,LXPQrZV4Cfw
2,"Patriots, Seahawks, Super Bowl 60 on Sunday in Santa Clara.",7.56,7.28,LXPQrZV4Cfw
3,A week from this Sunday.,14.84,1.30,LXPQrZV4Cfw
4,And how about the week that Kyle and Pete had?,16.14,3.08,LXPQrZV4Cfw


In [47]:
# dfs = []

# for transcript in tqdm(transcripts):
#     transcript_path = os.path.join(yt_data_path, transcript)
#     df = DataProcessing.load_from_file(path=transcript_path, file_type='csv')
#     dfs.append(df)

# raw_transcripts_df = DataProcessing.concat_dfs(dfs)
# raw_transcripts_df.head()

In [6]:
def clean_data(df) -> list[str]:
    """
    Split running text into sentence-like segments.
    Rules:
    - Split only where a period (.) or question mark (?) is immediately followed by
    whitespace. This avoids splitting decimals like ``4.5`` (digit after the dot,
    not a space).
    - Leading/trailing whitespace is stripped from each segment.
    - Empty segments are dropped.
    Caveat:
    - Abbreviations such as ``Mr. Smith`` still match ". " and may split incorrectly;
    handle those with a tokenizer or an allowlist if needed.
    Parameters
    ----------
    text : dataframe
        Full text to segment (e.g. one block of joined transcript lines).
    Returns
    -------
    list of str
        Non-empty strings, each intended as one sentence or clause.
    """

    text_joined = ' '.join(df.Text.to_list())
    raw_parts = re.split(r'(?<=[.?])\s+', text_joined)
    text_split = [p.strip() for p in raw_parts if p.strip()]

    return text_split

In [7]:
cleaned_transcripts = clean_data(raw_transcripts_df)
cleaned_transcripts

['We have reached the end.',
 'One last game.',
 'Patriots, Seahawks, Super Bowl 60 on Sunday in Santa Clara.',
 'A week from this Sunday.',
 'And how about the week that Kyle and Pete had?',
 'They both went 3-0.',
 'Both perfect conference championship.',
 'Great job, Pete.',
 'Best bets come through.',
 'Unreal.',
 'I got hot.',
 'Too late.',
 'Too little, too late.',
 'Yeah.',
 'Pete, the epic collapse is complete.',
 'Let me have my paper.',
 'I got something on the back of this paper.',
 'Look at Kyle in the postseason, 13-2 overall.',
 'You know what, man?',
 'You got to show up in the playoffs.',
 "And look, he's hit each of his last two best bets as well.",
 'That was what kind of strangled him in the regular season.',
 "He can't catch me with the best bets, so I have that going for me.",
 "It takes a while to figure out what we're looking at.",
 "I don't know if you can see this, but this says void.",
 'Yeah, it says void.',
 'You know why it says void?',
 'Why?',
 "We're voi

In [11]:
len(cleaned_transcripts)

374

## Prompt LLM

### Create Prompt

In [8]:
prompt = """
Role:
    You are a text analyst.

Task:
    Determine whether the input text contains a prediction, projection, or forecast.

Definitions:
    - A prediction is a statement that asserts or implies a future outcome, result, or event.
    - It may include confidence, uncertainty, or probability.
    - It must go beyond describing current or past facts.

Instructions:
    1. Read the text carefully.
    2. Decide if a prediction/projection/forecast is present.
    3. Assign:
        - 1 if a prediction is present
        - 0 if no prediction is present

Output format (strict JSON):
    {
        "label": 0 or 1
    }
"""

### Initialize LLMs

In [9]:
tgmf = TextGenerationModelFactory()

# Option 3: All NaviGator models
# models = tgmf.create_instances(tgmf.get_navigator_model_names())
models = tgmf.create_instances(['llama-3.1-8b-instant', 'llama-3.3-70b-versatile', 'openai/gpt-oss-120b'])
models

{'llama-3.1-8b-instant': <text_generation_models.LlamaInstantTextGenerationModel at 0x26ebf8bee10>,
 'llama-3.3-70b-versatile': <text_generation_models.LlamaVersatileTextGenerationModel at 0x26ebf963490>,
 'openai/gpt-oss-120b': <text_generation_models.GptOss120bTextGenerationModel at 0x26ebf8bd850>}

### Pass data + prompt into each LLM

In [10]:
results = []
PROMPT_TYPE = "zero-shot"   # change to "chain-of-thought" when needed

for cleaned_transcript_idx in range(len(cleaned_transcripts)):
    text = cleaned_transcripts[cleaned_transcript_idx]
    print(f"{cleaned_transcript_idx} --- Sentence: {text}")

    for model_name, model in models.items():
        raw_response, llm_label, llm_reasoning = llm_classify_text(
            data=text,
            base_prompt=prompt,
            prompt_type=PROMPT_TYPE,
            model=model
        )

        result = (
            text,
            raw_response,
            llm_label,
            llm_reasoning,
            model_name
        )
        results.append(result)

        if cleaned_transcript_idx < 3:
            if PROMPT_TYPE == "chain-of-thought":
                print(
                    f" Model: {model_name} "
                    f"| Label: {llm_label} "
                    f"| Reasoning: {llm_reasoning}"
                )
            else:
                print(
                    f" Model: {model_name} "
                    f"| Label: {llm_label}"
                )

0 --- Sentence: We have reached the end.
 Model: llama-3.1-8b-instant | Label: 0
 Model: llama-3.3-70b-versatile | Label: 0
 Model: openai/gpt-oss-120b | Label: 0
1 --- Sentence: One last game.
 Model: llama-3.1-8b-instant | Label: 0
 Model: llama-3.3-70b-versatile | Label: 0
 Model: openai/gpt-oss-120b | Label: 1
2 --- Sentence: Patriots, Seahawks, Super Bowl 60 on Sunday in Santa Clara.
 Model: llama-3.1-8b-instant | Label: 0
 Model: llama-3.3-70b-versatile | Label: 1
 Model: openai/gpt-oss-120b | Label: 1
3 --- Sentence: A week from this Sunday.
4 --- Sentence: And how about the week that Kyle and Pete had?
5 --- Sentence: They both went 3-0.
6 --- Sentence: Both perfect conference championship.
7 --- Sentence: Great job, Pete.
8 --- Sentence: Best bets come through.
9 --- Sentence: Unreal.
10 --- Sentence: I got hot.
11 --- Sentence: Too late.
12 --- Sentence: Too little, too late.
13 --- Sentence: Yeah.
14 --- Sentence: Pete, the epic collapse is complete.
15 --- Sentence: Let me 

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kbn2rg8gf1jax2p86k7vdraq` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99977, Requested 220. Please try again in 2m50.208s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

> - DataFrame below will have the same sentence (in text column) for all three llms (in the llm_name), then next set of rows will be another sentences with same llms.
> - llm_label column: {0: non-prediction, 1: prediction}

In [11]:
results_with_llm_label_df = pd.DataFrame(
    results,
    columns=['text', 'raw_response', 'llm_label', 'llm_reasoning', 'llm_name']
)

# Ensure labels are valid integers (0/1), fallback to -1 if malformed
results_with_llm_label_df['llm_label'] = (
    pd.to_numeric(results_with_llm_label_df['llm_label'], errors='coerce')
      .fillna(-1)
      .astype(int)
)

# Normalize reasoning: keep only when present (chain-of-thought), else None
results_with_llm_label_df['llm_reasoning'] = (
    results_with_llm_label_df['llm_reasoning'].where(
        results_with_llm_label_df['llm_reasoning'].notna(),
        None
    )
)

results_with_llm_label_df.tail(6)

,text,raw_response,llm_label,llm_reasoning,llm_name
273,"Hey, you made it all the way to the end.","{""label"": 0}",0,None,llama-3.1-8b-instant
274,"Hey, you made it all the way to the end.","{""label"": 0}",0,None,llama-3.3-70b-versatile
275,"Hey, you made it all the way to the end.","{""label"": 0}",0,None,openai/gpt-oss-120b
276,Thanks for that.,"{""label"": 0}",0,None,llama-3.1-8b-instant
277,Thanks for that.,"{""label"": 0}",0,None,llama-3.3-70b-versatile
278,Thanks for that.,"{""label"": 0}",0,None,openai/gpt-oss-120b


In [12]:
# Reshape to have llm names as column names
results_df = (
    results_with_llm_label_df
    .pivot_table(
        index='text',
        columns='llm_name',
        values='llm_label',
        aggfunc='first'   # one label per model per sentence
    )
    .reset_index()
    .rename(columns={'text': 'Base Sentence'})
)
results_df

llm_name,Base Sentence,llama-3.1-8b-instant,llama-3.3-70b-versatile,openai/gpt-oss-120b
0,And I can't wait to hear from all of you.,0,0,0
1,And I find it ironic that Adam Vinatieri was inducted into the Pro Football Hall of Fame.,0,0,0
2,And I think Seattle fans will be very happy.,1,1,1
3,And I want an MVP on top of it.,0,0,0
4,And I've been running an okey-doke week after week after week.,0,0,0
...,...,...,...,...
84,Yeah.,0,0,0
85,You both are big defensive MVPs.,0,0,0
86,You don't even get to score.,0,0,0
87,came down to a field goal by Adam Vinatieri to win that Super Bowl.,0,0,0


### Majority Vote (across LLMs)

> - llm_label column: {0: non-prediction, 1: prediction}

In [13]:
results_df['Majority Vote Label'] = (
    results_df
    .iloc[:, 1:]               # exclude sentence column
    .mode(axis=1)[0]
    .astype(int)
)

results_df.head()

llm_name,Base Sentence,llama-3.1-8b-instant,llama-3.3-70b-versatile,openai/gpt-oss-120b,Majority Vote Label
0,And I can't wait to hear from all of you.,0,0,0,0
1,And I find it ironic that Adam Vinatieri was inducted into the Pro Football Hall of Fame.,0,0,0,0
2,And I think Seattle fans will be very happy.,1,1,1,1
3,And I want an MVP on top of it.,0,0,0,0
4,And I've been running an okey-doke week after week after week.,0,0,0,0


Add 'Human Annotation', 'Human Reasoning' columns to write human input, and 'Video ID' column to grab the video source from the respective sentence.

In [14]:
results_df[['Human Annotation', 'Human Reasoning']] = np.nan
final_results_df = pd.concat([results_df, raw_transcripts_df[['Video ID']]], axis=1)
final_results_df.head()

,Base Sentence,llama-3.1-8b-instant,llama-3.3-70b-versatile,openai/gpt-oss-120b,Majority Vote Label,Human Annotation,Human Reasoning,Video ID
0,And I can't wait to hear from all of you.,0.0,0.0,0.0,0.0,NaN,NaN,fUmJAtFEGn8
1,And I find it ironic that Adam Vinatieri was inducted into the Pro Football Hall of Fame.,0.0,0.0,0.0,0.0,NaN,NaN,fUmJAtFEGn8
2,And I think Seattle fans will be very happy.,1.0,1.0,1.0,1.0,NaN,NaN,fUmJAtFEGn8
3,And I want an MVP on top of it.,0.0,0.0,0.0,0.0,NaN,NaN,fUmJAtFEGn8
4,And I've been running an okey-doke week after week after week.,0.0,0.0,0.0,0.0,NaN,NaN,fUmJAtFEGn8


In [15]:
predictions = final_results_df[final_results_df['Majority Vote Label']==1]

In [16]:
predictions

,Base Sentence,llama-3.1-8b-instant,llama-3.3-70b-versatile,openai/gpt-oss-120b,Majority Vote Label,Human Annotation,Human Reasoning,Video ID
2,And I think Seattle fans will be very happy.,1.0,1.0,1.0,1.0,NaN,NaN,fUmJAtFEGn8
5,"And at the end of the day, Super Bowl 60 is going to be won by the Seattle Seahawks.",1.0,1.0,1.0,1.0,NaN,NaN,fUmJAtFEGn8
6,"And he has been vocal quite a bit about Seattle being able to lay in wait and destroy teams, and that's where I'm going.",1.0,1.0,1.0,1.0,NaN,NaN,fUmJAtFEGn8
7,And it's going to be a 24-10 victory for the Seattle Seahawks.,1.0,1.0,1.0,1.0,NaN,NaN,fUmJAtFEGn8
8,And that's my prediction for Super Bowl 60 in advance.,1.0,1.0,1.0,1.0,NaN,NaN,fUmJAtFEGn8
11,And their defense is going to do it.,1.0,1.0,1.0,1.0,NaN,NaN,fUmJAtFEGn8
12,"And they are going to get the defensive stops, 24-10.",1.0,1.0,1.0,1.0,NaN,NaN,fUmJAtFEGn8
13,"And your Super Bowl MVP will be Demarcus Lawrence, baby.",1.0,1.0,1.0,1.0,NaN,NaN,fUmJAtFEGn8
17,But this is going to come down to an Andy Borogalis game-winning field goal.,1.0,1.0,1.0,1.0,NaN,NaN,fUmJAtFEGn8
34,I give them maybe 23-20.,1.0,1.0,1.0,1.0,NaN,NaN,fUmJAtFEGn8


In [17]:
base_data_path = DataProcessing.load_base_data_path(notebook_dir=notebook_dir)
save_data_path = os.path.join(base_data_path, "yt", "majority_vote", "sports")
DataProcessing.save_to_file(predictions, path=save_data_path, prefix='batch_2', save_file_type='csv', include_version=False)

Saving CSV file to: c:\Users\adria\OneDrive\Área de Trabalho\UF Data Studio\predictions\prediction_acquition-youtube\../data\yt\majority_vote\sports\batch_2.csv
